# 📖 Grays Sports Almanac
## Retrieval-Augmented Generation, in plain English

**Section 8** of *Search, Extract & Value: The Lightning Fast Assistant* 🚗💨

---

> ### 🖥️ This is the **local** version.
> The other version is **`06_almanac_api.ipynb`**.
>
> 📌 **These two notebooks are almost identical, and that is the point.**
> Everything in Acts 2 to 4 — reading your documents, turning them into
> numbers, searching them — happens **on your own machine either way.** Only
> the final answer is generated somewhere else. We come back to why that
> matters for a records office.

---

Biff's whole scheme was one book of facts. Not intelligence, not cleverness —
a reference he could look things up in, so he never had to guess.

That is Retrieval-Augmented Generation, and the name is far more complicated
than the idea:

> ### 🎯 Find the right page. Hand it to the model. Ask it to read.

The model is not searching. It is being handed the relevant page and asked to
answer from it — exactly what you would do with a new hire and a policy binder.

```mermaid
flowchart LR
    A["Your question"] --> B["Turn it into<br/>numbers"]
    C["Office policy<br/>documents"] --> D["Chop into<br/>sections"]
    D --> E["Turn each into<br/>numbers"]
    B --> F["Find the closest<br/>ones"]
    E --> F
    F --> G["Hand them to<br/>the model"]
    G --> H["Answer, with<br/>a citation"]
```

---

### What you need

| | |
|---|---|
| ⏱️ | About 8 minutes |
| 🎮 | A GPU with ~5 GB free |
| 📄 | The policy documents in `data/policy/` *(they ship with the repo)* |

---

# 🚦 LIVE DEMO PATH

## Setup

**This is the only cell where the two versions of this notebook differ.**

In [1]:
import json

import numpy as np

from delorean import agent, data, llm, rag, tools, viz

model, tokenizer = llm.load_local("qwen")
respond = agent.local_responder(model, tokenizer)

data.check()

Loading Qwen3.5 4B onto NVIDIA GeForce RTX 3080 ...


W0817 12:39:06.602000 40140 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Ready. Using 2.95 GB of 10.0 GB (4-bit NF4).


---

## Act 1 — 🔮 The question Section 4 got wrong

A taxpayer wants to split their lot. You need the minimum lot size. Ask the
model, with no tools and no documents — exactly as we did in Section 4.

In [2]:
QUESTION = ("In Hill Valley County, what is the minimum lot size to split an "
            "R-1 parcel?")

# The confident expert persona we wrote in Section 3.
EXPERT = (
    "You are a senior property appraiser with 25 years of experience in the "
    "Hill Valley County assessor's office. You know the county's parcels, "
    "policies, and sales history thoroughly. Answer confidently and concisely."
)

text, _calls = respond(
    [{"role": "system", "content": EXPERT},
     {"role": "user", "content": QUESTION}],
    [],          # no tools at all
)
print(text)

In Hill Valley County, the minimum lot size for an R-1 (Residential-1) parcel is **0.25 acres**.

Per the County's Zoning Ordinance, any subdivision of an R-1 parcel must result in lots that are at least 0.25 acres in size. This requirement is designed to maintain the agricultural and rural character of the county while allowing for necessary residential development. Any attempt to create a lot smaller than this threshold will be denied by the Planning Commission and the Board of Supervisors.


> ### ⚠️ There it is again.
>
> A specific number, attributed to an ordinance, with no hedging whatsoever. It
> reads exactly like the answer you wanted.
>
> **The real answer is 6,000 sq ft**, and it is written down in a document
> sitting on this machine. The model has simply never seen it.

---

## Act 2 — ✂️ Chop the manual into pieces

Four policy documents live in `data/policy/`. They are written the way a real
office writes them: numbered sections under headings.

So we split on the headings. Not on a fixed number of characters — **on the
sections the office already wrote.**

In [3]:
chunks = rag.load_chunks()

print(f"{len(chunks)} chunks from "
      f"{len(set(c['document'] for c in chunks))} documents\n")

for c in chunks[:5]:
    print(f"  [{c['document']} §{c['section']}]  {c['heading']}")
print("  ...")

27 chunks from 4 documents

  [assessment_calendar §2.1]  2.1 Assessment date
  [assessment_calendar §2.2]  2.2 The annual calendar
  [assessment_calendar §2.3]  2.3 The appeal deadline
  [assessment_calendar §2.4]  2.4 Informal review comes first
  [assessment_calendar §2.5]  2.5 Correcting errors outside the appeal window
  ...


> 📌 **Why headings and not every 500 characters?** Two reasons, and the second
> one matters more.
>
> 1. Fixed-size chunking cuts tables in half.
> 2. A chunk that begins *"4.3 Minimum lot size"* is one **a human can go and
>    check.** A chunk that begins mid-sentence on page four is not.
>
> An assessment office cannot act on an answer it cannot trace. So we throw
> away any piece that cannot name its own section — including each document's
> title block, which is why this is 27 chunks and not 31.

---

## Act 3 — 🔢 Text becomes numbers

This is the part everyone skips, and it is the only genuinely new idea in RAG.

Every chunk gets turned into a list of **384 numbers**. So does your question.
Passages that *mean* similar things end up with similar numbers — even when
they share almost no words.

In [4]:
vector = rag.embed(["Can I split my lot?"])[0]

print(f"'Can I split my lot?' becomes {len(vector)} numbers:\n")
print("  ", [round(v, 4) for v in vector[:8]], "...")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

'Can I split my lot?' becomes 384 numbers:

   [0.0066, -0.0825, 0.0492, -0.0422, -0.0012, -0.0591, -0.0289, -0.0809] ...


Meaningless on their own. The magic is in comparing them.

In [5]:
# The passage that actually answers the question.
target = next(c for c in chunks
              if c["document"] == "parcel_split_and_combination"
              and c["section"] == "4.3")

probes = [
    "What is the minimum lot size to split an R-1 parcel?",   # the real question
    "Can I divide my property into two?",                     # same idea, no shared words
    "How small is a lot allowed to be before it can be subdivided?",
    "What are the office's holiday hours?",                   # unrelated
    "Best pizza in Calgary",                                  # nonsense
]

target_vec = np.array(rag.embed([target["text"]])[0])
probe_vecs = np.array(rag.embed(probes))

print(f'Compared against:  {target["heading"]}\n')
for probe, vec in zip(probes, probe_vecs):
    score = float(vec @ target_vec)      # cosine, because the vectors are unit length
    bar = "#" * int(score * 40)
    print(f"  {score:5.3f}  {bar:<26}  {probe}")

Compared against:  4.3 Minimum lot size

  0.825  ################################  What is the minimum lot size to split an R-1 parcel?
  0.698  ###########################  Can I divide my property into two?
  0.748  #############################  How small is a lot allowed to be before it can be subdivided?
  0.432  #################           What are the office's holiday hours?
  0.406  ################            Best pizza in Calgary


> ### 🎯 Look at the second one.
>
> *"Can I divide my property into two?"* shares **no meaningful words** with
> *"minimum lot size"* — no "lot", no "split", no "R-1". It still scores high,
> because the numbers encode what the sentence *means*, not which words it
> used.
>
> That is why this beats keyword search for questions from the public, who do
> not know your terminology and should not have to.

> 📌 **The model doing the embedding is tiny** — `bge-small`, 33 million
> parameters, about 128 MB. It runs on the **CPU** in about a second for this
> whole corpus. You do not need a graphics card for the retrieval half of RAG,
> which is exactly why this notebook and its hosted twin behave identically here.

---

## Act 4 — 🔎 Find the right page

Put the chunks in a searchable store, embed the question, take the closest
matches. That is the whole retrieval step.

In [6]:
for hit in rag.search(QUESTION, top_k=3):
    print(f"  {hit['relevance']:.3f}  [{hit['document']} §{hit['section']}]  "
          f"{hit['heading']}")

  0.847  [parcel_split_and_combination §4.3]  4.3 Minimum lot size
  0.776  [parcel_split_and_combination §4.2]  4.2 Outstanding taxes bar a split
  0.691  [parcel_split_and_combination §4.4]  4.4 Additional requirements


In [7]:
top = rag.search(QUESTION, top_k=1)[0]
print(top["text"][:700])

### 4.3 Minimum lot size

Each parcel resulting from a split must independently satisfy the minimum lot
area for its zoning district:

| Zoning district | Minimum lot area | Typical neighborhoods |
|---|---|---|
| R-1 Single Family | 6,000 sq ft | Hilldale, Lyon Estates, Courthouse Square |
| R-1A Single Family, large lot | 12,000 sq ft | Twin Pines Ranch |
| R-2 Two Family | 7,500 sq ft | Eastwood Ravine |
| R-3 Multi Family | 10,000 sq ft | scattered |
| C-1 Commercial | 8,000 sq ft | Courthouse Square |

A parcel that would leave any resulting lot below the minimum shall be denied
regardless of the tax status.

---


> ### 💡 There is the answer, in black and white. **6,000 sq ft.**
>
> Nothing has generated anything yet. We have simply found the right page —
> and we can name it: *Policy 4.3*.

---

## Act 5 — 🧰 It is just a tool

Here is the part that surprises people.

We do not need a new kind of agent, a "RAG pipeline", or any new machinery.
Retrieval is **a function that takes a question and returns text** — which is
the definition of a tool from Section 6.

So it goes in the toolbelt, next to the one we already had.

In [8]:
viz.show_toolbelt(["lookup_parcel"])                              # Section 6
viz.show_toolbelt(["lookup_parcel", "search_policy"],
                  active="search_policy")                          # Section 8

In [9]:
# Same dispatch as any other tool. Nothing special about it.
out = json.loads(tools.dispatch("search_policy",
                                {"question": "minimum lot size for a split"}))

for passage in out["data"]["passages"][:2]:
    print(f"  {passage['cite']:<42} relevance {passage['relevance']}")

  parcel_split_and_combination §4.3          relevance 0.827
  parcel_split_and_combination §4.4          relevance 0.728


> ### 🎯 That is the whole of Section 8.
>
> RAG is not a different kind of intelligence. It is **a tool that reads your
> filing cabinet**, handed to the same agent, through the same loop, in the
> same list.

---

## Act 6 — 📖 Now ask again

Same question as Act 1. Same model. This time it can reach the manual.

In [10]:
office_agent = agent.build_agent(respond,
                                 tool_names=["lookup_parcel", "search_policy"])

state = agent.ask(office_agent, QUESTION)

viz.show_trace(state["trace"],
               answer=agent.final_answer(state),
               question=QUESTION)

> ### ⚖️ Put that next to Act 1.
>
> | | Act 1 | Act 6 |
> |---|---|---|
> | Answer | invented / refused | **6,000 sq ft** |
> | Source | none | **Policy 4.3** |
> | Can you check it? | no | **open the file** |
>
> In Section 4 this model cited a Planning Commission that does not exist. Now
> it quotes a section number you can go and read. That difference — *"here is
> where I got it"* — is the difference between a toy and something an office
> can actually use.

---

## 🎬 That's Section 8

1. 📄 **Chunk on the structure the document already has.** A citation a human
   can look up is worth more than a slightly better similarity score.
2. 🔢 **Embeddings capture meaning, not keywords.** Which is why the public can
   ask in their words instead of yours.
3. 🧰 **RAG is a tool.** Same agent, same loop, one more entry in the list.
4. 📌 **Always return the citation.** An answer an appraiser cannot trace is an
   answer they cannot use.

> ### 🔒 And one thing worth telling your IT department
>
> Everything in Acts 2 to 4 ran **on this machine** — reading the documents,
> chunking them, embedding them, searching them. That is true in the hosted
> version of this notebook too.
>
> Your document index never leaves the building. Only the handful of passages
> the search actually returns are sent to the model. **You can keep the whole
> corpus in-house and still use a hosted model.**

> ### ⏭️ **Next: Section 9 — The DeLorean Engine.**
> All of this, on a model small enough to run on the machine under your desk.

---

---

# 🔬 EXPLORE ON YOUR OWN

---

## 7. 🕳️ What happens when the manual has no answer

Retrieval **always** returns its closest matches, even when nothing is
relevant. It has no way to say "there is nothing here". Watch the relevance
scores collapse.

In [11]:
for question in ["Can a parcel be split if taxes are owed?",     # covered
                 "What are the office's holiday hours?",          # not covered
                 "How do I renew a dog licence?"]:                # not remotely
    best = rag.search(question, top_k=1)[0]
    print(f"  {best['relevance']:.3f}  {question}")
    print(f"          -> best match: {best['heading']}\n")

  0.803  Can a parcel be split if taxes are owed?
          -> best match: 4.2 Outstanding taxes bar a split

  0.569  What are the office's holiday hours?
          -> best match: 2.2 The annual calendar

  0.564  How do I renew a dog licence?
          -> best match: 2.5 Correcting errors outside the appeal window



> ⚠️ **A low score is a signal you have to act on yourself.** Nothing in the
> retrieval step will stop the model answering confidently from a passage that
> has nothing to do with the question. In production you would set a floor —
> below it, tell the user the manual does not cover it.

---

## 8. 📂 Point it at your own documents

`data/policy/` holds markdown, but the loader takes **PDFs** too. Drop your
office's real manual in that folder and re-index — no code changes.

In [12]:
for path in data.policy_documents():
    print(" ", path.name)

print("\nAdd a .pdf or .md here and re-run rag.build_index() to include it.")

  assessment_calendar.md
  parcel_split_and_combination.md
  ratio_study_standards.md
  sales_qualification.md

Add a .pdf or .md here and re-run rag.build_index() to include it.


---

## 9. ⚙️ The embedding model is one line

We chose `BAAI/bge-small-en-v1.5`: MIT licensed, 33M parameters, 384
dimensions, and fast enough on a CPU that no graphics card is needed.

Swap `rag.EMBEDDING_MODEL` for a bigger one if your corpus is larger or your
questions are subtler. Bigger is not automatically better — measure it against
questions your office actually gets.

In [13]:
print("model    :", rag.EMBEDDING_MODEL)
print("device   :", rag.EMBEDDING_DEVICE)
print("dimension:", len(rag.embed(["test"])[0]))

model    : BAAI/bge-small-en-v1.5
device   : cpu
dimension: 384


---

<div align="center">

### 🕰️ *"With that book I can't lose!"* 🕰️
### *…as long as you cite it.*

🚗💨 📖 🏘️

</div>